In [20]:
import torch
import torch.nn as nn
from gensim.models import Word2Vec
import numpy as np
from sklearn.model_selection import train_test_split
import re
from pathlib import Path
from torch.utils.data import DataLoader, Dataset

In [2]:
# Dictionary mapping contractions to their full forms
contractions_dict = {
    "he's": "he is",
    "i'm": "I am",
    "you're": "you are",
    "we've": "we have",
    "they've": "they have",
    "don't": "do not",
    "isn't": "is not",
    "it's": "it is",
    "didn't": "did not",
    "aren't": "are not",
    "let's": "let us",
    "couldn't": "could not",
    "wasn't": "was not",
    "weren't": "were not",
    "ain't": "am not",
    "i've": "I have",
    "that's": "that is",
    "i'll": "I will",
    "you'd": "you would",
    "they're": "they are",
    "i won't": "I will not",
    "can't": "cannot",
    "you've": "you have",
    "there's": "there is",
    "won't": "will not",
    "you'll": "you will",
    "doesn't": "does not",
    "must've": "must have",
    "what's": "what is",
    "we're": "we are",
    "haven't": "have not",
    "wouldn't": "would not",
    "i'd": "I would",
    "she's": "she is",
    "nobody's": "nobody is",
    "we'll": "we will",
    "they'd": "they would",
    "mustn't": "must not",
    "could've": "could have",
    "shouldn't": "should not",
    "he'll": "he will",
    "he'd": "he would",
    "hadn't": "had not",
    "where'd": "where did",
    "we'd": "we would",
}

# Function to replace contractions in the text
def replace_contractions(text, contractions_map):
    # Create a regex pattern that matches any of the contractions
    pattern = re.compile(r'\b(' + '|'.join(re.escape(key) for key in contractions_map.keys()) + r')\b')
    # Replace contractions using the dictionary
    return pattern.sub(lambda x: contractions_map[x.group()], text)

# Function to process the text file
def process_text_file(input_file, output_file, contractions_map):
    # Read the contents of the input file
    with open(input_file, 'r') as file:
        text = file.read().lower()

    # Replace contractions
    new_text = replace_contractions(text, contractions_map)

    # Write the modified text to the output file
    with open(output_file, 'w') as file:
        file.write(new_text)

    return new_text

# Specify the input and output file paths
input_file = 'adele.txt'  # Replace with the actual file path
output_file = 'output.txt'

# Process the text file if not present
if not Path(output_file).exists():
    modified_texte = process_text_file(input_file, output_file, contractions_dict)
    print("Contractions replaced and saved to", output_file)
else: 
    with open(output_file, "r") as f:
        modified_texte = f.read().lower()


# Parameters

In [14]:
# It is arbitrary values
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
N_LAYERS = 2
DROPOUT = 0.5
N_EPOCHS = 10
LR = 3e-4
BATCH_SIZE = 32
SEQ_LEN = 30

# Tokenization

In [133]:
file_path = "output.txt"

with open(file_path, 'r') as file:
    text = file.readlines()

text = modified_texte.split("\n")

sentences = []
for line in text:
    # print(line)

    # Remove all non-alphanumeric characters and convert to lowercase
    clean_line = re.sub(r'[^\w\s]', '', line.lower())
    
    # TODO: faire en sorte que ca soit propre, y a des phrase d'un seul mot et c'est vraiment nul
    #       J me suis pas concentré sur ca pour le moment. J'ai aussi l'impression que la ponctuation reste dans le bail donc chelou
    
    words = clean_line.split() 

    if len(words) > 1:
        # Add the EOS token at the end of sentence
        # words.insert(0,"<SOS>")
        words.append("<EOS>")
        sentences.append(words)
    else:
        print("Phrase ignored:", words)
print(sentences[0])
# Vector size of 100, it can be modified, we can play with the parameters of Word2Vec
word2vec_model = Word2Vec(sentences, vector_size=EMBEDDING_DIM, window=30, min_count=1, epochs=100)
word2vec_model.save("word2vec100_adele.model")

word_vector = word2vec_model.wv["the"]
inverse_word_vector = word2vec_model.wv.most_similar(positive=[word_vector], topn=1)
print(inverse_word_vector)
print(word_vector)

Phrase ignored: ['sweetest']
Phrase ignored: ['sweetest']
Phrase ignored: ['sweetest']
Phrase ignored: ['darling']
Phrase ignored: ['ooh']
Phrase ignored: ['depleted']
Phrase ignored: ['hustle']
['looking', 'for', 'some', 'education', '<EOS>']
[('the', 1.0)]
[-0.26401383 -0.27510187 -0.07486645  1.3615438   0.6525396  -1.0053154
  0.23676622  0.88090354 -0.087955    1.1056484   0.39801744 -0.35549533
  0.83330977 -0.8175831   0.7441132  -0.41861543  1.2380409  -0.75890195
 -1.6977874   2.244582   -0.1656194   1.8549392   0.43831497 -0.10537397
 -0.468257   -0.3650151   0.9413912   0.02164089 -0.9850431  -0.96560043
 -1.2025408   0.04473543 -0.37846357  0.12522006  0.34479305  0.43513107
  0.87279314 -0.45115417 -0.17630468  0.59911084 -0.48450232 -0.20055382
  1.246013    0.3819775   0.14482108 -0.41901752  1.0272084  -1.0200543
  0.72360474 -0.6583328   0.9322932  -0.8491292   0.3330956   0.58358157
 -0.15980826  0.8023833   0.28007668  0.28774643  0.18928319  0.11847808
 -0.4531824  

In [5]:
print(word2vec_model.wv.most_similar("baby"))

[('details', 0.5388527512550354), ('bore', 0.5336499810218811), ('tone', 0.5048681497573853), ('miss', 0.5046097636222839), ('setting', 0.4743771553039551), ('signs', 0.4680875837802887), ('give', 0.460506796836853), ('lights', 0.4517943561077118), ('off', 0.44779136776924133), ('read', 0.4276643693447113)]


In [18]:
sentences

[['looking', 'for', 'some', 'education', '<EOS>'],
 ['made', 'my', 'way', 'into', 'the', 'night', '<EOS>'],
 ['all', 'that', 'bullshit', 'conversation', '<EOS>'],
 ['baby',
  'cannot',
  'you',
  'read',
  'the',
  'signs',
  'i',
  'will',
  'not',
  'bore',
  'you',
  'with',
  'the',
  'details',
  'baby',
  '<EOS>'],
 ['i', 'do', 'not', 'even', 'wanna', 'waste', 'your', 'time', '<EOS>'],
 ['let', 'us', 'just', 'say', 'that', 'maybe', '<EOS>'],
 ['you', 'could', 'help', 'me', 'ease', 'my', 'mind', '<EOS>'],
 ['i',
  'am',
  'not',
  'mr',
  'right',
  'but',
  'if',
  'you',
  'are',
  'looking',
  'for',
  'fast',
  'love',
  '<EOS>'],
 ['if', 'that', 'is', 'love', 'in', 'your', 'eyes', '<EOS>'],
 ['it', 'is', 'more', 'than', 'enough', '<EOS>'],
 ['had', 'some', 'bad', 'love', '<EOS>'],
 ['so',
  'fast',
  'love',
  'is',
  'all',
  'that',
  'i',
  'have',
  'got',
  'on',
  'my',
  'mind',
  'ooh',
  'ooh',
  '<EOS>'],
 ['ooh', 'ooh', 'looking', 'for', 'some', 'affirmation', '<EO

In [134]:
data = []
for s in sentences:
    for w in s:
        data.append(word2vec_model.wv[w])
        # data.append(w)

data = torch.tensor(np.array(data))

def create_sequence(data, seq_len):
    n = len(data)
    X = []
    y = []
    for i in range(n - seq_len - 1):
        X.append(data[i:i+seq_len])
        y.append(data[i+1:i+seq_len+1])
    
    return X, y

train_X, train_y = create_sequence(data, SEQ_LEN)

class Text(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_dataset = Text(train_X, train_y)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [135]:
np.array(train_X).shape, np.array(train_y).shape

((22401, 30, 100), (22401, 30, 100))

In [62]:
def sentences_to_vectors(sentences, word2vec_model):
    """Convert the sentences to the vector learned by word2vec

    Args:
        sentences (List[List[str]]): The first list contain the lines/sentences, the second one contain the words of the sentences
        word2vec_model (Word2Vec): Word2Vec object trained on the actual corpus

    Returns:
        List[List[torch.Tensor]]: The same 2 list with the words converted to their word2Vec vector
    """
    indices = []
    for sentence in sentences:
        sentence_vectors = []
        for word in sentence: # Si le mot est dans le vocabulaire
            if word in word2vec_model.wv.key_to_index:  
                sentence_vectors.append(word2vec_model.wv[word])
            else: # Si le mot n'existe pas dans le vocabulaire
                print(f"Le mot '{word}' n'existe pas dans le vocabulaire.")
                exit(0) 
        indices.append(torch.tensor(np.array(sentence_vectors)))
    return indices

# Conversion des phrases en vecteur
sentence_vectors = sentences_to_vectors(sentences, word2vec_model)

print(f"Nb of sentences: {len(sentence_vectors)}")
print(f"Nb of words in first sentence: {len(sentence_vectors[0])}")
print(f"Embbedding size of first word: {len(sentence_vectors[0][0])}")

# Séparer les données en ensembles d'entraînement et de validation
train_vectors, val_vectors = train_test_split(sentence_vectors, test_size=0.2, random_state=42)
train_sentences, val_sentences = train_test_split(sentences, test_size=0.2, random_state=42)

def check_mapping(train_vectors, val_vectors, train_sentences, val_sentences):
    """ Check via the size that all vectors are correctly mapped to the right sentence

    Args:
        train_vectors (List[List[torch.Tensor]]): list of vector of check
        val_vectors (List[List[torch.Tensor]]): list of vector of check
        train_sentences (List[List[str]): list of words to check
        val_sentences (List[List[str]): list of words to check

    Raises:
        Exception: if the check fails
    """
    test_vectors = [train_vectors, val_vectors]
    test_sentences = [train_sentences, val_sentences]

    for i in range(len(test_vectors)):
        for index in range(len(test_vectors[i])):
            if (len(test_vectors[i][index]) != len(test_sentences[i][index])):
                raise Exception("The size of the vector isn't the same that the corresponding sentence")

check_mapping(train_vectors, val_vectors, train_sentences, val_sentences)

Nb of sentences: 2393
Nb of words in first sentence: 5
Embbedding size of first word: 100


In [63]:
vocab_size = len(word2vec_model.wv.key_to_index)
print(vocab_size)

1345


# Model

In [64]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers):
        super(LSTM, self).__init__()

        self.lstm = nn.LSTM(embedding_dim, vocab_size + hidden_dim, n_layers, proj_size= vocab_size)

    def forward(self, x):
        x, _  = self.lstm(x)
        # print(x.shape)
        return x
    
lstm = LSTM(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS)

In [72]:
class NLP(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers, dropout, model_type='LSTM'):
        super(NLP, self).__init__()
        self.vocab_size = vocab_size
        self.num_layers = n_layers
        self.rnn_type = model_type
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.hidden_dim = hidden_dim

        if model_type == 'LSTM':
            self.nlp = nn.LSTM(embedding_dim, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        elif model_type == 'GRU':
            self.nlp = nn.GRU(embedding_dim, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        else:
            raise Exception("Model type not supported")
        self.fc = nn.Linear(hidden_dim, embedding_dim)
        
    def forward(self, x, hidden):
        # x = [batch_size, seq_len, embed_dim]
        # print(x.shape)
        x, hidden = self.nlp(x)
        # x = [batch_size, seq_len, hidden_dim]
        # print(x.shape)
        x = self.fc(x)
        # x = [batch_size, seq_len, embed_dim]
        # print(x.shape)
        return x, hidden
    
    def init_hidden(self, batch_size):
        if self.rnn_type == 'LSTM':
            # LSTM requires both hidden state and cell state
            hidden = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(self.device)
            cell = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(self.device)
            return (hidden, cell)
        else:
            # GRU only requires the hidden state
            hidden = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(self.device)
            return hidden

In [ ]:
def train(model, vectors, sentences, n_epochs, lr, batch_size, seq_len, name):
    # Setup GPU related variables
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device = {device}")
    torch.cuda.empty_cache()
    model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(reduction='mean')

    model.train()

    for epoch in range(n_epochs):
        train_losses = []
        for i, sentence in enumerate(sentences):
            hidden = model.init_hidden(batch_size)

            # Substract the last word as we didn't include the eos token (so we couldn't predict it)
            x = vectors[i]
            
            # Create the one hot encoding of the correct prediction
            y = torch.tensor(np.zeros((len(x), vocab_size)))
            
            # Start the sequence to the first word as we didn't include the sos token (so we couldn't predict it)
            for j in range(0, len(x)):
                y[0][word2vec_model.wv.get_index(sentence[j])] = 1.0

            optimizer.zero_grad()
            #print(sentence)
            #print(x.shape)
            output, hidden = model(x, hidden)

            print(output.shape, y.shape)

            loss = criterion(output, y)
            train_losses.append(loss.cpu().detach())
            loss.backward()
            optimizer.step()

            if i % 100 == 0:
                print(f"Epoch {epoch}, step {i}, loss {loss.item()}")
        
        print(f"Epoch {epoch} finished. Train loss: {np.array(train_losses).mean()}, Perplexity: {np.exp(np.array(train_losses).mean())}")
        
    torch.save(model.state_dict(), f"model_save/{name}.pth")

In [142]:
def train(model, dataloader, n_epochs, lr, batch_size, seq_len, name):
    # Setup GPU related variables
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device = {device}")
    torch.cuda.empty_cache()
    model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(reduction='mean')

    model.train()

    for epoch in range(n_epochs):
        train_losses = []
        for i, (X, y) in enumerate(dataloader):
            # print(X.shape)
            hidden = model.init_hidden(batch_size)
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            output, hidden = model(X, hidden)

            # print(word2vec_model.wv.most_similar(positive=[last_w_y.cpu().detach().numpy()], topn=1))
            # print(word2vec_model.wv.most_similar(positive=[last_w_output.cpu().detach().numpy()], topn=1))

            # output = output.view(-1, vocab_size)
            # print(output.shape, y.shape)
            # y = y.view(-1, vocab_size)

            # print(output.shape, y.shape)
            loss = criterion(output, y)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            if i % 100 == 0:
                print(f"Epoch {epoch}, step {i}, loss {loss.item()}")
        
        print(f"Epoch {epoch} finished. Train loss: {np.array(train_losses).mean()}, Perplexity: {np.exp(np.array(train_losses).mean())}")

    torch.save(model.state_dict(), f"model_save/{name}.pth")

In [40]:
#model_type = 'GRU'
#GRU_model = NLP(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, model_type)

#train(GRU_model, train_vectors, train_sentences, N_EPOCHS, LR, BATCH_SIZE, SEQ_LEN, f"{model_type}_model_Word2Vec{EMBEDDING_DIM}")

In [141]:
model_type = "LSTM"
LSTM_model = NLP(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, model_type)

# train(LSTM_model, train_vectors, train_sentences, N_EPOCHS, LR, BATCH_SIZE, SEQ_LEN, f"{model_type}_model_Word2Vec{EMBEDDING_DIM}")
train(LSTM_model, train_loader, N_EPOCHS, LR, BATCH_SIZE, SEQ_LEN, f"{model_type}_model_Word2Vec{EMBEDDING_DIM}")

device = cpu
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])
Epoch 0, step 0, loss 3.3806095123291016
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])
torch.Size([32, 30, 100]) torch.Size([32, 30, 100])


KeyboardInterrupt: 

In [ ]:
def generate_text(model, encoder: Word2Vec, start_word, max_words=20, random_sample=False):
    """
    Generate text based on the trained model output.
    
    Parameters:
    - model: The trained PyTorch model.
    - encoder: The OneHotEncoder used for encoding the words.
    - start_word: The initial word to start generating text.
    - max_words: Number of words to generate.
    - random_sample: If True, sample from the distribution instead of taking the max probability.
    
    Returns:
    - generated_text: The generated sequence of words.
    """
    model.eval()

    start_word = start_word.lower().split()
    # Initialize the generated text with the start word
    generated_words = start_word
    
    # Convert the start word to its one-hot encoded representation
    input_tensor = []
    
    for word in start_word:
        input_tensor.append(encoder.wv.get_vector(word))
        
    input_tensor = torch.Tensor(input_tensor).unsqueeze(0).to(model.device)  # Add batch dimension

    # Initialize hidden state
    hidden = model.init_hidden(batch_size=1)
    
    # Generate the specified number of words
    while(True):
        # Get the model output with the hidden state
        with torch.no_grad():
            output, hidden = model(input_tensor, hidden)  # Pass hidden state
    
        # Apply softmax to get probabilities
        probabilities = torch.softmax(output, dim=-1).squeeze().cpu().numpy()
        
        # Resize probabilities in case of single word given as input
        if len(probabilities.shape) != 2:
            probabilities = probabilities.reshape((1,-1))

        for probability in probabilities:
            # Determine the next word
            if random_sample:
                next_index = np.random.choice(len(probability), p=probability)
            else:
                next_index = np.argmax(probability)

            # Get the corresponding word from the encoder
            next_word = encoder.wv.index_to_key[next_index]

            stop = False
            if next_word == "<EOS>" or len(generated_words) > max_words:
                stop = True
                break
            else:   
                generated_words.append(next_word)

        if stop:
            break
        
        # Update the input tensor with the new word
        input_tensor = [encoder.wv.get_vector("<SOS>")]
        for word in generated_words:
            input_tensor.append(encoder.wv.get_vector(word))
        
        input_tensor = torch.Tensor(input_tensor).unsqueeze(0).to(model.device)  # Add batch dimension
    
    # Join the generated words into a single string
    generated_text = ' '.join(generated_words)
    return generated_text

In [131]:
def generate_text(model, encoder, start_word, num_words=10, random_sample=False):
    """
    Generate text based on the trained model output.
    
    Parameters:
    - model: The trained PyTorch model.
    - encoder: The OneHotEncoder used for encoding the words.
    - start_word: The initial word to start generating text.
    - num_words: Number of words to generate.
    - random_sample: If True, sample from the distribution instead of taking the max probability.
    
    Returns:
    - generated_text: The generated sequence of words.
    """
    model.eval()

    start_word = start_word.lower().split()
    # Initialize the generated text with the start word
    generated_words = start_word
    
    # Convert the start word to its one-hot encoded representation
    input_tensor = []
    
    for word in start_word:
        input_tensor.append(encoder.wv.get_vector(word))
    input_tensor = torch.Tensor(input_tensor).unsqueeze(0).to(model.device)  # Add batch dimension

    # Initialize hidden state
    hidden = model.init_hidden(batch_size=1)
    
    # Generate the specified number of words
    for _ in range(num_words):
        # Get the model output with the hidden state
        with torch.no_grad():
            output, hidden = model(input_tensor, hidden)  # Pass hidden state

        # print(output.shape)
        # test = word2vec_model.wv.most_similar(positive=[output], topn=1)
        # print(test)
        # Apply softmax to get probabilities
        probabilities = torch.softmax(output, dim=-1).squeeze().cpu().numpy()

        # print(probabilities.shape)

        if len(probabilities.shape) != 2:
            probabilities = probabilities.reshape((1,-1))

        # taking the last word predicted because is the new word of the sentence
        proba = probabilities[-1]
        # print(proba)
        proba = word2vec_model.wv.most_similar(positive=[proba], topn=10)
        words = [word for word, _ in proba]
        values = [value for _, value in proba]
        # print(proba)
        # print(proba.shape)

        if random_sample:
            next_index = np.random.choice(len(values), p=nn.Sigmoid()(torch.tensor(np.array(values))))
        else:
            next_index = np.argmax(values)

        next_word = words[next_index]

        # print(next_word)

        if next_word == "<EOS>":
            print("Le token <EOS> a été atteint")
            break

        generated_words.append(next_word)

        # Update the input tensor with the new word => increasing the context
        # input_tensor = input_tensor[:,1:,:] # => if we want to remove the first word and keep the same context length => not good selon moi
        new_input = encoder.wv.get_vector(next_word)
        # print(new_input.shape, input_tensor.shape)
        new_input = torch.Tensor(new_input).unsqueeze(0).unsqueeze(0).to(model.device)  # Add batch dimension
        # print(new_input.shape, input_tensor.shape)
        input_tensor = torch.cat((input_tensor, new_input), dim=1)
        # print(input_tensor.shape)
    
    # Join the generated words into a single string
    generated_text = ' '.join(generated_words)
    return generated_text

In [132]:
# Exemple d'utilisation
generated_text = generate_text(LSTM_model, word2vec_model, start_word='who i', random_sample=False)
print(generated_text)

generated_text = generate_text(LSTM_model, word2vec_model, start_word='had some', random_sample=False)
print(generated_text)

generated_text = generate_text(LSTM_model, word2vec_model, start_word='every time he am', random_sample=False)
print(generated_text)

generated_text_random = generate_text(LSTM_model, word2vec_model, start_word='Baby', random_sample=True)
print(generated_text_random)

who i am am streets unkind unkind unkind unkind unkind unkind unkind
had some unkind unkind unkind unkind unkind unkind unkind unkind unkind unkind
every time he am around unkind unkind unkind unkind unkind unkind unkind unkind unkind


ValueError: probabilities do not sum to 1